# Code Generator

The requirement: use a Frontier model to generate high performance C++ code from Python code


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">Reminder: OPTIONAL to execute C++ code or Rust code</h2>
            <span style="color:#f71;">As an alternative, you can run it on the website given yesterday</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h1 style="color:#900;">Important Note</h1>
            <span style="color:#900;">
            In this lab, I use high end models GPT 5, Claude 4.5 Sonnet, Gemini 2.5 Pro, Grok 4, which are the slightly higher priced models. The costs are still low, but if you'd prefer to keep costs ultra low, please pick lower cost models like gpt-5-nano.
            </span>
        </td>
    </tr>
</table>

In [5]:
# imports

import os
import io
import sys
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display


In [6]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
zai_api_key = os.getenv('ZAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if zai_api_key:
    print(f"Z.AI API Key exists and begins {zai_api_key[:7]}")
else:
    print("Z.AI API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:6]}")
else:
    print("OpenRouter API Key not set (and this is optional)")



OpenAI API Key not set
Z.AI API Key exists and begins 8ed45f4
Google API Key exists and begins AI
OpenRouter API Key exists and begins sk-or-


In [7]:
# Connect to client libraries

zai_url       = "https://api.z.ai/api/paas/v4/"
openrouter_url = "https://openrouter.ai/api/v1"
gemini_url    = "https://generativelanguage.googleapis.com/v1beta/openai/"

openai     = OpenAI(api_key=zai_api_key,       base_url=zai_url)
zai        = OpenAI(api_key=zai_api_key,       base_url=zai_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url) if openrouter_api_key else None
gemini     = OpenAI(api_key=google_api_key,    base_url=gemini_url)     if google_api_key    else None


In [8]:
models = [
    "glm-4.7",
    "glm-5",
    "anthropic/claude-sonnet-4-6",
    "openai/gpt-4.1-mini",
    "deepseek/deepseek-chat-v3-0324",
    "qwen/qwen3-coder:free",
]

clients = {
    "glm-4.7":                        zai,
    "glm-5":                          zai,
    "anthropic/claude-sonnet-4-6":     openrouter,
    "openai/gpt-4.1-mini":            openrouter,
    "deepseek/deepseek-chat-v3-0324": openrouter,
    "qwen/qwen3-coder:free":          openrouter,
}


In [9]:
from system_info import retrieve_system_info, rust_toolchain_info

system_info = retrieve_system_info()
rust_info = rust_toolchain_info()
rust_info

{'installed': False,
 'rustc': {'path': '',
  'version': '',
  'host_triple': '',
  'release': '',
  'commit_hash': ''},
 'cargo': {'path': '', 'version': ''},
 'rustup': {'path': '',
  'version': '',
  'active_toolchain': '',
  'default_toolchain': '',
  'toolchains': [],
  'targets_installed': []},
 'rust_analyzer': {'path': ''},
 'env': {'CARGO_HOME': '',
  'RUSTUP_HOME': 'C:\\Users\\BOSS\\.rustup',
  'RUSTFLAGS': '',
  'CARGO_BUILD_TARGET': ''},
 'execution_examples': []}

In [10]:
message = f"""
Here is a report of the system information for my computer.
I want to run a Rust compiler to compile a single rust file called main.rs and then execute it in the simplest way possible.
Please reply with whether I need to install a Rust toolchain to do this. If so, please provide the simplest step by step instructions to do so.

If I'm already set up to compile Rust code, then I'd like to run something like this in Python to compile and execute the code:
```python
compile_command = # something here - to achieve the fastest possible runtime performance
compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
run_command = # something here
run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
return run_result.stdout
```
Please tell me exactly what I should use for the compile_command and run_command.
Have the maximum possible runtime performance in mind; compile time can be slow. Fastest possible runtime performance for this platform is key.
Reply with the commands in markdown.

System information:
{system_info}

Rust toolchain information:
{rust_info}
"""

response = openai.chat.completions.create(model=models[0], messages=[{"role": "user", "content": message}])
display(Markdown(response.choices[0].message.content))

Based on the system information provided, you do **not** have a Rust toolchain installed (`'installed': False`). To compile and run Rust code, you must install the toolchain first.

Since you are on Windows 11 and have `winget` available, this is the simplest installation method.

### Step 1: Install the Rust Toolchain

Open your terminal (PowerShell or Command Prompt) and run the following command:

```powershell
winget install --id Rustlang.Rustup -e
```

After the installation completes, **close and reopen** your terminal to ensure the path variables are updated.

### Step 2: Python Script for Maximum Runtime Performance

Since your goal is the **fastest possible runtime performance** for your AMD Ryzen 9 7900X on Windows, we will use `rustc` directly with specific optimization flags. We will enable `native` CPU tuning to utilize the specific instruction sets of your processor (like AVX2/AVX512), aggressive link-time optimization (LTO), and the highest optimization level.

Here is the Python code:

```python
import subprocess
import os
import sys

# Define the output executable name
exe_name = "main.exe"

# The compile command utilizes the following flags for maximum performance:
# -C opt-level=3          : The highest level of general optimization.
# -C target-cpu=native     : Tuning the code specifically for your AMD Ryzen 9 7900X hardware.
# -C lto=fat               : Performs Link Time Optimization across the whole binary.
# -C codegen-units=1       : Disables parallel codegen to allow for better global optimization.
compile_command = [
    "rustc",
    "-C", "opt-level=3",
    "-C", "target-cpu=native",
    "-C", "lto=fat",
    "-C", "codegen-units=1",
    "main.rs",
    "-o", exe_name
]

# The run command executes the generated executable.
# On Windows, we use '.\\' to ensure it runs from the current directory.
run_command = [f".\\{exe_name}"]

try:
    print(f"Compiling with flags: {' '.join(compile_command[2:-2])}...")
    compile_result = subprocess.run(compile_command, check=True, text=True, capture_output=True)
    
    print("Running executable...")
    run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
    
    print("Output:")
    print(run_result.stdout)

except subprocess.CalledProcessError as e:
    print(f"An error occurred:\n{e.stderr}")
```

### Explanation of Flags used:
*   **`-C target-cpu=native`**: This is the most critical flag for your request. It tells the compiler to generate instructions specific to your Ryzen 9 processor, rather than generic x86_64 instructions.
*   **`-C lto=fat`**: "Fat" Link Time Optimization analyzes the entire program at once to optimize across function boundaries, resulting in a faster binary at the cost of slower compilation.
*   **`-C codegen-units=1`**: By default, Rust compiles in parallel to speed up build times. Setting this to 1 removes that limit, allowing the optimizer to see the whole program as a single unit for better inlining and vectorization decisions.

## For C++, overwrite this with the commands from yesterday, or for Rust, use the new commands

Or just use the website like yesterday:

 https://www.programiz.com/cpp-programming/online-compiler/

In [11]:
# Rust not installed — compiling as C++ with g++ instead
compile_command = ["g++", "-std=c++17", "-O3", "-march=native", "main.cpp", "-o", "main.exe"]
run_command = ["main.exe"]


## And now, on with the main task

In [12]:
language = "C++" # switched from Rust (not installed)
extension = "cpp"

system_prompt = f"""
Your task is to convert Python code into high performance {language} code.
Respond only with {language} code. Do not provide any explanation other than occasional comments.
The {language} response needs to produce an identical output in the fastest possible time.
"""

def user_prompt_for(python):
    return f"""
Port this Python code to {language} with the fastest possible implementation that produces identical output in the least time.
The system information is:
{system_info}
Your response will be written to a file called main.{language} and then compiled and executed; the compilation command is:
{compile_command}
Respond only with {language} code.
Python code to port:

```python
{python}
```
"""

In [13]:
def messages_for(python):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(python)}
    ]
 

In [14]:
def write_output(code):
    with open(f"main.{extension}", "w") as f:
        f.write(code)

In [15]:
def port(model, python):
    client = clients[model]
    reasoning_effort = "high" if 'gpt' in model else None
    response = client.chat.completions.create(model=model, messages=messages_for(python), reasoning_effort=reasoning_effort)
    reply = response.choices[0].message.content
    reply = reply.replace('```cpp','').replace('```rust','').replace('```','')
    return reply

In [16]:
def run_python(code):
    globals_dict = {"__builtins__": __builtins__}

    buffer = io.StringIO()
    old_stdout = sys.stdout
    sys.stdout = buffer

    try:
        exec(code, globals_dict)
        output = buffer.getvalue()
    except Exception as e:
        output = f"Error: {e}"
    finally:
        sys.stdout = old_stdout

    return output

In [17]:
# Use the commands from GPT 5

def compile_and_run(code):
    write_output(code)
    try:
        subprocess.run(compile_command, check=True, text=True, capture_output=True)
        run_result = subprocess.run(run_command, check=True, text=True, capture_output=True)
        return run_result.stdout
    except subprocess.CalledProcessError as e:
        return f"An error occurred:\n{e.stderr}"

In [18]:
python_hard = """# Be careful to support large numbers

def lcg(seed, a=1664525, c=1013904223, m=2**32):
    value = seed
    while True:
        value = (a * value + c) % m
        yield value
        
def max_subarray_sum(n, seed, min_val, max_val):
    lcg_gen = lcg(seed)
    random_numbers = [next(lcg_gen) % (max_val - min_val + 1) + min_val for _ in range(n)]
    max_sum = float('-inf')
    for i in range(n):
        current_sum = 0
        for j in range(i, n):
            current_sum += random_numbers[j]
            if current_sum > max_sum:
                max_sum = current_sum
    return max_sum

def total_max_subarray_sum(n, initial_seed, min_val, max_val):
    total_sum = 0
    lcg_gen = lcg(initial_seed)
    for _ in range(20):
        seed = next(lcg_gen)
        total_sum += max_subarray_sum(n, seed, min_val, max_val)
    return total_sum

# Parameters
n = 10000         # Number of random numbers
initial_seed = 42 # Initial seed for the LCG
min_val = -10     # Minimum value of random numbers
max_val = 10      # Maximum value of random numbers

# Timing the function
import time
start_time = time.time()
result = total_max_subarray_sum(n, initial_seed, min_val, max_val)
end_time = time.time()

print("Total Maximum Subarray Sum (20 runs):", result)
print("Execution Time: {:.6f} seconds".format(end_time - start_time))
"""

In [25]:
from styles import CSS

with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to {language}") as ui:
    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python = gr.Code(
                label="Python (original)",
                value=python_hard,
                language="python",
                lines=26
            )
        with gr.Column(scale=6):
            cpp = gr.Code(
                label=f"{language} (generated)",
                value="",
                language="cpp",
                lines=26
            )

    with gr.Row(elem_classes=["controls"]):
        python_run = gr.Button("Run Python", elem_classes=["run-btn", "py"])
        model = gr.Dropdown(models, value=models[0], show_label=False)
        convert = gr.Button(f"Port to {language}", elem_classes=["convert-btn"])
        cpp_run = gr.Button(f"Run {language}", elem_classes=["run-btn", "cpp"])

    with gr.Row(equal_height=True):
        with gr.Column(scale=6):
            python_out = gr.TextArea(label="Python result", lines=8, elem_classes=["py-out"])
        with gr.Column(scale=6):
            cpp_out = gr.TextArea(label=f"{language} result", lines=8, elem_classes=["cpp-out"])

    convert.click(fn=port, inputs=[model, python], outputs=[cpp])
    python_run.click(fn=run_python, inputs=[python], outputs=[python_out])
    cpp_run.click(fn=compile_and_run, inputs=[cpp], outputs=[cpp_out])

ui.launch(inbrowser=True)


C:\Users\BOSS\AppData\Local\Temp\ipykernel_138720\3949588770.py:3: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to {language}") as ui:
C:\Users\BOSS\AppData\Local\Temp\ipykernel_138720\3949588770.py:3: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Monochrome(), title=f"Port from Python to {language}") as ui:


* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


## RESULTS!

GLM 4.7: PASS (2nd Slowest)
GLM 5: PASS (Slowest) 
DeepSeek Chat v3: FAIL  
Qwen3 Coder 30B: FAIL  
Claude Sonnet 4.6: PASS (Fastest)    
GPT-4.1 Mini: PASS (2nd Fastest)    

3rd place: GLM 4.7  
2nd place: GPT-4.1 Mini 
**1st place: Claude Sonnet 4.6**  

In [26]:
print(f"In Ed's experimenet, the GPT-OSS 120B model outcome is {33.755209/0.000304:,.0f} times faster than the Python code.")

In Ed's experimenet, the GPT-OSS 120B model outcome is 111,037 times faster than the Python code.
